# Huấn luyện & Tuning DCGAN (Giai đoạn 4 — Mục 2.3 & 3.1)

Notebook này dùng để chạy huấn luyện và thử nghiệm các cấu hình DCGAN trên **Google Colab** hoặc **Kaggle GPU (T4/P100)**.

Các bước thực hiện:
1. Kiểm tra GPU & đồng bộ code mới nhất từ GitHub (`feature/gan-data-pipeline`).
2. Chuẩn bị dữ liệu (Fashion-MNIST / CIFAR-10 tự tải; CelebA giải nén từ Google Drive hoặc gắn Kaggle Dataset).
3. **Giai đoạn 0 (Overfit Check):** Chạy nhanh kiểm chứng luồng train & lưu checkpoint.
4. **Giai đoạn 1 (Tuning ngắn):** Chạy thử nghiệm 4 cấu hình (`base`, `lr_0001`, `latent_128`, `no_smoothing`).
5. **Theo dõi trực quan:** Vẽ biểu đồ Loss $G$ / $D$ và hiển thị lưới ảnh mẫu $8 \times 8$ để kiểm tra Mode Collapse.
6. Sao lưu checkpoint & metrics ra Google Drive (lưu ngoài Git).

In [ ]:
# 1. Kiểm tra GPU hiện có
!nvidia-smi

In [ ]:
# 2. Mount Google Drive (nếu chạy trên Google Colab) để lưu checkpoint & đọc CelebA
import os
IN_COLAB = 'COLAB_GPU' in os.environ or os.path.exists('/content')
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

In [ ]:
# 3. Clone hoặc Pull code mới nhất từ GitHub
REPO_URL = "https://github.com/pthanhlam2016-oss/Model-Image-Generator-.git"  # Sửa nếu dùng repo khác
BRANCH = "feature/gan-data-pipeline"
REPO_DIR = "/content/Model-Image-Generator-" if IN_COLAB else "/kaggle/working/Model-Image-Generator-"

if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git pull origin {BRANCH}

### Thiết lập Tham số Chạy & Chuẩn bị Dữ liệu
Chọn `DATASET` (`fashion_mnist`, `cifar10`, `celeba`) và `EXP_NAME` (`base`, `lr_0001`, `latent_128`, `no_smoothing`).

In [ ]:
DATASET = "fashion_mnist"   # "fashion_mnist" | "cifar10" | "celeba"
EXP_NAME = "base"           # "base" | "lr_0001" | "latent_128" | "no_smoothing"
STAGE = "tuning"            # "overfit" | "tuning" | "full"
EPOCHS = 15                 # Số epoch cho giai đoạn tuning ngắn (full thì để None để lấy theo YAML)
RESUME_CKPT = ""            # Điền đường dẫn checkpoint_latest.pt nếu cần chạy tiếp

CONFIG_PATH = f"configs/dcgan/{DATASET}/{EXP_NAME}.yaml"
print(f"Cấu hình được chọn: {CONFIG_PATH}")

In [ ]:
# (Chỉ dành cho CelebA trên Colab) Giải nén dữ liệu CelebA từ Drive vào datasets/raw/celeba
if DATASET == "celeba" and IN_COLAB:
    if not os.path.exists("datasets/raw/celeba/img_align_celeba"):
        !mkdir -p datasets/raw/celeba
        !unzip -q "/content/drive/MyDrive/Celebs_Zip_Data/archive (2).zip" -d "datasets/raw/celeba"
        print("Đã giải nén xong CelebA!")
    else:
        print("Dữ liệu CelebA đã sẵn sàng.")

### Giai đoạn 0: Kiểm tra Overfit (Mục 2.3 — Pass/Fail)
Train lặp lại trên tập nhỏ cố định 256 ảnh (trong phạm vi 64–512 ảnh) để chứng minh code đúng và mô hình đủ capacity.

In [ ]:
!python scripts/train_dcgan.py --config {CONFIG_PATH} --overfit_samples 256 --epochs 30 --stage overfit --run_name overfit_{DATASET}_{EXP_NAME}

### Giai đoạn 1 / 2: Huấn luyện DCGAN trên GPU
Chạy lệnh huấn luyện chính thức. Nếu bị ngắt giữa chừng, điền đường dẫn `checkpoint_latest.pt` vào `RESUME_CKPT` ở trên rồi chạy lại cell này.

In [ ]:
cmd = f"python scripts/train_dcgan.py --config {CONFIG_PATH} --stage {STAGE} --run_name dcgan_{DATASET}_{EXP_NAME}"
if EPOCHS is not None:
    cmd += f" --epochs {EPOCHS}"
if RESUME_CKPT:
    cmd += f" --resume {RESUME_CKPT}"

print("Đang chạy:", cmd)
!{cmd}

### Trực quan hóa Loss & Lưới ảnh mẫu (Theo dõi Mode Collapse)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

loss_csv = f"outputs/metrics/dcgan/{DATASET}/{EXP_NAME}/loss.csv"
sample_img = f"outputs/samples/dcgan/{DATASET}/{EXP_NAME}/latest_sample.png"

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

if os.path.exists(loss_csv):
    df = pd.read_csv(loss_csv)
    axes[0].plot(df['epoch'], df['loss_d'], label='Loss D (Discriminator)', marker='o')
    axes[0].plot(df['epoch'], df['loss_g'], label='Loss G (Generator)', marker='s')
    axes[0].set_title(f'Biểu đồ Loss: {DATASET} ({EXP_NAME})')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

if os.path.exists(sample_img):
    img = Image.open(sample_img)
    axes[1].imshow(img)
    axes[1].set_title('Lưới ảnh mẫu Fixed Noise (8x8)')
    axes[1].axis('off')

plt.tight_layout()
plt.show()

### Sao lưu Checkpoint & Logs ra Google Drive (Lưu ngoài Git)

In [ ]:
if IN_COLAB:
    DRIVE_BACKUP = f"/content/drive/MyDrive/BTL_DL_DCGAN_Outputs/{DATASET}/{EXP_NAME}"
    !mkdir -p "{DRIVE_BACKUP}"
    !cp -r outputs/checkpoints/dcgan/{DATASET}/{EXP_NAME} "{DRIVE_BACKUP}/checkpoints"
    !cp -r outputs/samples/dcgan/{DATASET}/{EXP_NAME} "{DRIVE_BACKUP}/samples"
    !cp -r outputs/metrics/dcgan/{DATASET}/{EXP_NAME} "{DRIVE_BACKUP}/metrics"
    print(f"Đã sao lưu toàn bộ kết quả vào Google Drive: {DRIVE_BACKUP}")